# Discriminant Analysis for Alzheimer's Disease

This notebook mirrors the structure of INFO381A_LR.ipynb, but focuses on linear discriminant analysis (LDA), feature separation, and statistical interpretation.

Imports

In [ ]:
import json
import os
from pathlib import Path

import kagglehub as kh
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2, f_oneway
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid')

OUT_DIR = Path('outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

REDUCED_EXCLUDE_COLS = [
    'MMSE', 'ADL', 'FunctionalAssessment',
    'MemoryComplaints', 'BehavioralProblems',
    'Confusion', 'Disorientation', 'PersonalityChanges',
    'DifficultyCompletingTasks', 'Forgetfulness',
]

Load dataset

In [ ]:
path = kh.dataset_download('rabieelkharoua/alzheimers-disease-dataset')
print('Path to dataset files:', path)

files = os.listdir(path)
print('Files in directory:', files)

csv_file = [f for f in files if f.endswith('.csv')][0]
csv_path = os.path.join(path, csv_file)

DATA_PATH = Path(csv_path)

df = pd.read_csv(csv_path)
print('Shape:', df.shape)
df.head()

Basic overview

In [ ]:
df.info()
df.describe().T

Check for missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
missing.sort_values(ascending=False)

Class distribution

In [ ]:
print(df['Diagnosis'].value_counts())
print(df['Diagnosis'].value_counts(normalize=True))

plt.figure(figsize=(6, 4))
sns.countplot(x='Diagnosis', data=df, palette='magma')
plt.title('Diagnosis Distribution')
plt.xlabel('Diagnosis')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

Drop irrelevant columns and define feature groups

In [ ]:
df = df.drop(columns=['PatientID', 'DoctorInCharge'], errors='ignore')

target_col = 'Diagnosis'
binary_cols = [
    'Gender', 'Smoking', 'FamilyHistoryAlzheimers',
    'CardiovascularDisease', 'Diabetes', 'Depression',
    'HeadInjury', 'Hypertension',
    'MemoryComplaints', 'BehavioralProblems',
    'Confusion', 'Disorientation',
    'PersonalityChanges', 'DifficultyCompletingTasks',
    'Forgetfulness'
]
categorical_cols = ['Ethnicity', 'EducationLevel']
numeric_cols = [col for col in df.columns if col not in binary_cols + categorical_cols + [target_col]]

print('Numeric:', numeric_cols)
print('Binary:', binary_cols)
print('Categorical:', categorical_cols)

X = df.drop(columns=[target_col]).copy()
y = df[target_col].astype(int).copy()
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

FULL_FEATURES = X.columns.tolist()
REDUCED_FEATURES = X.drop(columns=REDUCED_EXCLUDE_COLS, errors='ignore').columns.tolist()

analysis_df = pd.concat([X, y.rename(target_col)], axis=1)

print(f'Full feature count: {len(FULL_FEATURES)}')
print(f'Reduced feature count: {len(REDUCED_FEATURES)}')

Correlation matrix

In [ ]:
plot_cols = [col for col in FULL_FEATURES if col in analysis_df.columns][:30]
plt.figure(figsize=(13, 10))
sns.heatmap(analysis_df[plot_cols].corr(), cmap='magma', center=0)
plt.title('Correlation Matrix (First 30 Full-Set Features)')
plt.tight_layout()
plt.show()

Prepare analysis helpers

In [ ]:
def make_pipeline():
    return Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('lda', LinearDiscriminantAnalysis()),
        ]
    )


def compute_wilks_lambda(X: pd.DataFrame, y: pd.Series):
    prep = Pipeline(
        steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]
    )
    x_mat = prep.fit_transform(X)
    y_arr = np.asarray(y)
    classes = np.unique(y_arr)

    n_samples, n_features = x_mat.shape
    n_groups = len(classes)

    grand_mean = np.mean(x_mat, axis=0)
    centered = x_mat - grand_mean
    total_sscp = centered.T @ centered

    within_sscp = np.zeros_like(total_sscp)
    for cls in classes:
        x_group = x_mat[y_arr == cls]
        group_mean = np.mean(x_group, axis=0)
        group_centered = x_group - group_mean
        within_sscp += group_centered.T @ group_centered

    log_lambda = None
    ridge_used = None
    for ridge in (1e-10, 1e-8, 1e-6, 1e-4, 1e-2):
        identity = np.eye(n_features)
        sign_w, logdet_w = np.linalg.slogdet(within_sscp + ridge * identity)
        sign_t, logdet_t = np.linalg.slogdet(total_sscp + ridge * identity)
        if sign_w > 0 and sign_t > 0:
            log_lambda = float(logdet_w - logdet_t)
            ridge_used = ridge
            break

    if log_lambda is None:
        return {
            'wilks_lambda': None,
            'log_wilks_lambda': None,
            'chi_square_approx': None,
            'df': int(n_features * (n_groups - 1)),
            'p_value_approx': None,
            'ridge_used': None,
        }

    wilks_lambda = float(np.exp(log_lambda))
    df = int(n_features * (n_groups - 1))

    bartlett_factor = (n_samples - 1) - (n_features + n_groups) / 2
    chi_square = float(max(0.0, -bartlett_factor * log_lambda))
    p_value = float(1 - chi2.cdf(chi_square, df)) if df > 0 else None

    return {
        'wilks_lambda': wilks_lambda,
        'log_wilks_lambda': log_lambda,
        'chi_square_approx': chi_square,
        'df': df,
        'p_value_approx': p_value,
        'ridge_used': ridge_used,
    }


def run_lda_task(df: pd.DataFrame, target: str, features: list[str], task_name: str):
    available = [col for col in features if col in df.columns]
    data = df[available + [target]].dropna(subset=[target]).copy()

    X = data[available]
    y = data[target].astype(int)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pipeline = make_pipeline()
    y_pred = cross_val_predict(pipeline, X, y, cv=cv)
    y_proba = cross_val_predict(pipeline, X, y, cv=cv, method='predict_proba')

    accuracy = accuracy_score(y, y_pred)
    macro_f1 = f1_score(y, y_pred, average='macro')
    roc_curve_data = None
    if y.nunique() == 2:
        roc_auc = roc_auc_score(y, y_proba[:, 1])
        fpr, tpr, _ = roc_curve(y, y_proba[:, 1])
        roc_curve_data = {
            'fpr': fpr.tolist(),
            'tpr': tpr.tolist(),
            'auc': float(roc_auc),
        }
    else:
        roc_auc = roc_auc_score(y, y_proba, multi_class='ovr', average='macro')
    wilks_stats = compute_wilks_lambda(X, y)

    fitted = make_pipeline()
    fitted.fit(X, y)
    lda = fitted.named_steps['lda']

    explained_var = None
    cumsum_explained_var = None
    if hasattr(lda, 'explained_variance_ratio_'):
        explained_var = lda.explained_variance_ratio_.tolist()
        cumsum_explained_var = float(np.cumsum(lda.explained_variance_ratio_)[-1]) if len(lda.explained_variance_ratio_) > 0 else 0.0

    if y.nunique() == 2 and hasattr(lda, 'coef_'):
        feature_strength = np.abs(lda.coef_.ravel())
    else:
        feature_strength = np.mean(np.abs(lda.scalings_), axis=1)

    importance = (
        pd.DataFrame({'feature': available, 'importance_abs': feature_strength})
        .sort_values('importance_abs', ascending=False)
        .reset_index(drop=True)
    )

    importance_file = OUT_DIR / f'lda_feature_importance_{task_name}.csv'
    importance.to_csv(importance_file, index=False)

    return {
        'task': task_name,
        'target': target,
        'n_samples': int(len(data)),
        'n_features': int(len(available)),
        'random_state': 42,
        'cv_accuracy': float(accuracy),
        'cv_macro_f1': float(macro_f1),
        'cv_roc_auc': float(roc_auc),
        'roc_curve': roc_curve_data,
        'explained_variance_ratio': explained_var,
        'total_explained_variance': cumsum_explained_var,
        'wilks_lambda': wilks_stats,
        'top_features': importance.head(10).to_dict(orient='records'),
        'importance_file': str(importance_file),
    }


def education_group_difference(df: pd.DataFrame, features: list[str]):
    target = 'EducationLevel'
    data = df[features + [target]].dropna(subset=[target]).copy()

    groups = sorted(data[target].astype(int).unique().tolist())
    results = []

    for feature in features:
        if feature not in data.columns:
            continue
        grouped = [data.loc[data[target] == g, feature].dropna().values for g in groups]
        if any(len(arr) < 2 for arr in grouped):
            continue
        if np.allclose(np.nanstd(data[feature].values), 0):
            continue

        f_stat, p_value = f_oneway(*grouped)

        grand_mean = data[feature].mean()
        ss_between = sum(len(arr) * (np.mean(arr) - grand_mean) ** 2 for arr in grouped)
        ss_total = np.sum((data[feature].values - grand_mean) ** 2)
        eta_sq = ss_between / ss_total if ss_total > 0 else 0.0

        group_means = {f'edu_{g}_mean': float(np.mean(arr)) for g, arr in zip(groups, grouped)}

        row = {
            'feature': feature,
            'f_stat': float(f_stat),
            'p_value': float(p_value),
            'eta_sq': float(eta_sq),
        }
        row.update(group_means)
        results.append(row)

    effects = pd.DataFrame(results).sort_values('eta_sq', ascending=False)
    effects_file = OUT_DIR / 'education_group_differences_anova.csv'
    effects.to_csv(effects_file, index=False)

    return {
        'n_features_tested': int(len(effects)),
        'top_effects': effects.head(10).to_dict(orient='records'),
        'effects_file': str(effects_file),
    }


def plot_top_importance(result: dict, title: str, top_n: int = 10):
    importance = pd.read_csv(result['importance_file']).head(top_n).iloc[::-1]
    plt.figure(figsize=(10, 6))
    plt.barh(importance['feature'], importance['importance_abs'], color='#3b6fb6')
    plt.title(title)
    plt.xlabel('Absolute discriminant strength')
    plt.tight_layout()
    plt.show()


def plot_roc_curve_comparison(full_result: dict, reduced_result: dict):
    plt.figure(figsize=(8, 6))
    for result, color in ((full_result, '#2a9d8f'), (reduced_result, '#e76f51')):
        roc_data = result.get('roc_curve')
        if not roc_data:
            continue
        plt.plot(
            roc_data['fpr'],
            roc_data['tpr'],
            color=color,
            linewidth=2.5,
            label=f"{result['task']} (AUC = {roc_data['auc']:.3f})",
        )

    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5, label='Chance')
    plt.title('ROC Curves: Full vs Reduced LDA Models')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

Diagnosis (Full Feature Set)

In [ ]:
full_result = run_lda_task(
    df=analysis_df,
    target='Diagnosis',
    features=FULL_FEATURES,
    task_name='diagnosis_full_feature_set',
)

full_result

In [ ]:
plot_top_importance(full_result, 'Top LDA Features: Diagnosis (Full Set)')

Diagnosis (Reduced Feature Set)

In [ ]:
reduced_result = run_lda_task(
    df=analysis_df,
    target='Diagnosis',
    features=REDUCED_FEATURES,
    task_name='diagnosis_reduced_feature_set',
)

reduced_result

Reduced-Set Feature Importance

In [ ]:
plot_top_importance(reduced_result, 'Top LDA Features: Diagnosis (Reduced Set)')

Metric Delta (Full vs Reduced)

In [ ]:
metric_delta = {
    'accuracy_delta_full_minus_reduced': full_result['cv_accuracy'] - reduced_result['cv_accuracy'],
    'macro_f1_delta_full_minus_reduced': full_result['cv_macro_f1'] - reduced_result['cv_macro_f1'],
    'roc_auc_delta_full_minus_reduced': full_result['cv_roc_auc'] - reduced_result['cv_roc_auc'],
}
metric_delta

Summary Table

In [ ]:
results_df = pd.DataFrame([
    {
        'task': full_result['task'],
        'target': full_result['target'],
        'n_samples': full_result['n_samples'],
        'n_features': full_result['n_features'],
        'cv_accuracy': full_result['cv_accuracy'],
        'cv_macro_f1': full_result['cv_macro_f1'],
        'cv_roc_auc': full_result['cv_roc_auc'],
        'wilks_lambda': full_result['wilks_lambda']['wilks_lambda'],
        'wilks_p_value': full_result['wilks_lambda']['p_value_approx'],
    },
    {
        'task': reduced_result['task'],
        'target': reduced_result['target'],
        'n_samples': reduced_result['n_samples'],
        'n_features': reduced_result['n_features'],
        'cv_accuracy': reduced_result['cv_accuracy'],
        'cv_macro_f1': reduced_result['cv_macro_f1'],
        'cv_roc_auc': reduced_result['cv_roc_auc'],
        'wilks_lambda': reduced_result['wilks_lambda']['wilks_lambda'],
        'wilks_p_value': reduced_result['wilks_lambda']['p_value_approx'],
    },
]).round(4)

display(results_df)

Performance Comparison

In [ ]:
plot_df = results_df[['task', 'cv_accuracy', 'cv_macro_f1', 'cv_roc_auc']].set_index('task')

ax = plot_df.plot(kind='bar', figsize=(10, 6), color=['#2a9d8f', '#e76f51', '#264653'])
ax.set_title('LDA Performance: Full vs Reduced Feature Sets')
ax.set_ylabel('Score')
ax.set_xlabel('Experiment')
ax.set_ylim(0, 1)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

ROC Curve Comparison

In [ ]:
plot_roc_curve_comparison(full_result, reduced_result)

Write report

In [ ]:
report = {
    'diagnosis_full_feature_set': full_result,
    'diagnosis_reduced_feature_set': reduced_result,
    'feature_set_definition': {
        'full_feature_count': len(FULL_FEATURES),
        'reduced_feature_count': len(REDUCED_FEATURES),
        'reduced_excluded_columns': REDUCED_EXCLUDE_COLS,
    },
}

report_path = OUT_DIR / 'discriminant_analysis_report.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(f'Wrote: {report_path}')

Conclusion

The LDA results summarize how strongly symptoms, clinical measurements, and demographic variables separate diagnosis, gender, and education groups in this dataset.